In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path(r"/mnt/c/dev/my_ml_project")

OOF_DIR = PROJECT_ROOT / "oof_preds"
SUB_DIR = PROJECT_ROOT / "submissions"
SEED_SUB_DIR = SUB_DIR / "seed_ensemble"
SEED_SUB_DIR.mkdir(parents=True, exist_ok=True)

def rank01(x):
    return rankdata(x) / len(x)

def load_npy(path):
    path = Path(path)
    print(path, "exists:", path.exists())
    return np.load(path)

# =========================
# y
# =========================
y_stack = np.load(OOF_DIR / "combo_te_v1_s10_seed2024" / "y_stack.npy")

# =========================
# seed42 component OOF
# 기존 combo_te_s10 파일명 기준
# =========================
main_42 = load_npy(OOF_DIR / "combo_te_s10" / "combo_cat_oof_seed.npy")
shallow_42 = load_npy(OOF_DIR / "combo_te_s10" / "combo_cat_shallow_oof.npy")
random_42 = load_npy(OOF_DIR / "combo_te_s10" / "combo_cat_random_oof.npy")
xgb_42 = load_npy(OOF_DIR / "combo_te_s10" / "combo_xgb_oof.npy")

# =========================
# seed2024 component OOF
# =========================
main_2024 = load_npy(OOF_DIR / "combo_te_v1_s10_seed2024" / "main_cat_oof.npy")
shallow_2024 = load_npy(OOF_DIR / "combo_te_v1_s10_seed2024" / "shallow_cat_oof.npy")
random_2024 = load_npy(OOF_DIR / "combo_te_v1_s10_seed2024" / "random_cat_oof.npy")
xgb_2024 = load_npy(OOF_DIR / "combo_te_v1_s10_seed2024" / "xgb_oof.npy")

# =========================
# seed777 component OOF
# =========================
main_777 = load_npy(OOF_DIR / "combo_te_v1_s10_seed777" / "main_cat_oof.npy")
shallow_777 = load_npy(OOF_DIR / "combo_te_v1_s10_seed777" / "shallow_cat_oof.npy")
random_777 = load_npy(OOF_DIR / "combo_te_v1_s10_seed777" / "random_cat_oof.npy")
xgb_777 = load_npy(OOF_DIR / "combo_te_v1_s10_seed777" / "xgb_oof.npy")

/mnt/c/dev/my_ml_project/oof_preds/combo_te_s10/combo_cat_oof_seed.npy exists: True
/mnt/c/dev/my_ml_project/oof_preds/combo_te_s10/combo_cat_shallow_oof.npy exists: True
/mnt/c/dev/my_ml_project/oof_preds/combo_te_s10/combo_cat_random_oof.npy exists: True
/mnt/c/dev/my_ml_project/oof_preds/combo_te_s10/combo_xgb_oof.npy exists: True
/mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_s10_seed2024/main_cat_oof.npy exists: True
/mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_s10_seed2024/shallow_cat_oof.npy exists: True
/mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_s10_seed2024/random_cat_oof.npy exists: True
/mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_s10_seed2024/xgb_oof.npy exists: True
/mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_s10_seed777/main_cat_oof.npy exists: True
/mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_s10_seed777/shallow_cat_oof.npy exists: True
/mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_s10_seed777/random_cat_oof.npy exists: True
/mnt/c/dev/my_ml_project/oof_pre

In [3]:
# component별 3-seed 평균
main_avg = (main_42 + main_2024 + main_777) / 3
shallow_avg = (shallow_42 + shallow_2024 + shallow_777) / 3
random_avg = (random_42 + random_2024 + random_777) / 3
xgb_avg = (xgb_42 + xgb_2024 + xgb_777) / 3

main_rank = rank01(main_avg)
shallow_rank = rank01(shallow_avg)
random_rank = rank01(random_avg)
xgb_rank = rank01(xgb_avg)

# 기존 고정 weight 기준 점수
fixed_blend = (
    0.44 * main_rank +
    0.08 * shallow_rank +
    0.35 * random_rank +
    0.13 * xgb_rank
)

print("fixed component-avg blend:", roc_auc_score(y_stack, fixed_blend))

fixed component-avg blend: 0.7407420942412384


In [4]:
best_score = 0
best_weights = None
best_blend = None

for w_main in np.arange(0.30, 0.61, 0.01):
    for w_shallow in np.arange(0.00, 0.21, 0.01):
        for w_random in np.arange(0.20, 0.56, 0.01):
            w_xgb = 1 - w_main - w_shallow - w_random

            if w_xgb < 0:
                continue

            blend = (
                w_main * main_rank +
                w_shallow * shallow_rank +
                w_random * random_rank +
                w_xgb * xgb_rank
            )

            score = roc_auc_score(y_stack, blend)

            if score > best_score:
                best_score = score
                best_weights = (w_main, w_shallow, w_random, w_xgb)
                best_blend = blend.copy()

print("best_score:", best_score)
print("best_weights:", best_weights)

print("current best 3-seed final avg:", 0.7407465066345057)
print("improvement:", best_score - 0.7407465066345057)

best_score: 0.7407584925303079
best_weights: (np.float64(0.37000000000000005), np.float64(0.0), np.float64(0.5400000000000003), np.float64(0.08999999999999964))
current best 3-seed final avg: 0.7407465066345057
improvement: 1.1985895802202329e-05


In [5]:
np.save(
    OOF_DIR / "combo_te_v1_s10_seed42_2024_777_component_weighted_oof.npy",
    best_blend
)

pd.DataFrame([{
    "score": best_score,
    "w_main": best_weights[0],
    "w_shallow": best_weights[1],
    "w_random": best_weights[2],
    "w_xgb": best_weights[3],
}]).to_csv(
    OOF_DIR / "combo_te_v1_s10_seed42_2024_777_component_weighted_summary.csv",
    index=False
)